# When is a complex detector justified?

This evidence-synthesis notebook predefines one economic comparison: CUSUM versus GLR and BOCPD on independent, low-variance-signal gradual-drift streams with zero label delay, action cost 0.6 and retraining cost 0.05.

Evidence is sufficient only if the benefit is both statistically resolved and economically material, while the false-alarm rate satisfies a separate constraint. Null streams are independent of the threshold-calibration streams and the gradual-drift evaluation streams.

Smoke execution establishes software correctness only. It must never be presented as scientific confirmation.

## Primary question and decision protocol

For a candidate method (A), compute on each matched simulation stream (r):

[
\Delta_r(A)=R_r(A)-R_r(\mathrm{CUSUM}).
]

Negative (Delta) favours the candidate. A material gain is predeclared as at least 0.01 units of average net regret per deployment observation.

We allocate the 5% familywise inferential error budget equally between economic effects and independent null false-alarm constraints. Each family is Bonferroni-adjusted for two candidate detectors. Bootstrap percentile bounds are approximate and must not be treated as universally valid simultaneous confidence intervals.

The declared minimum is 100 independent drift streams and 100 independent null streams, not 100 time observations. A winner is never selected solely because it has the lowest observed average regret.

In [ ]:
import os
from IPython.display import display
from comparative_change_detection import ComparisonDesign, run_comparison
from evidence_synthesis import (
    PrimaryProtocol, preregistered_verdict, descriptive_context
)

SMOKE = os.getenv("SYNTHESIS_SMOKE", "0") == "1"
REPEATS = int(os.getenv("SYNTHESIS_REPEATS", "100"))
NULL_STREAMS = int(os.getenv("SYNTHESIS_NULL_STREAMS", "199"))
if REPEATS < 1 or NULL_STREAMS < 2:
    raise ValueError("Invalid independent-stream counts")
design = ComparisonDesign(
    n_fit=80 if SMOKE else 160,
    n_cal=35 if SMOKE else 80,
    n_live=45 if SMOKE else 120,
    null_streams=19 if SMOKE else NULL_STREAMS,
    delays=(0,),
    refit_window=70 if SMOKE else 100,
    scheduled_every=10 if SMOKE else 15,
    glr_max_window=16 if SMOKE else 32,
    action_costs=(0.6,),
    refit_costs=(0.05,),
)
protocol = PrimaryProtocol(bootstrap_draws=100 if SMOKE else 2000)
calibration, models, economics = run_comparison(
    design, evaluation_repeats=1 if SMOKE else REPEATS,
    scenarios=("low_variance_signal",),
    regimes=("stable", "gradual_drift"),
    seed=2026,
)
display(calibration)

## Predeclared verdicts and uncertainty

This analysis returns 'supported_in_simulation' only when the corrected upper bound of mean regret difference is lower than the negative material-gain threshold, the simultaneous false-alarm upper bound is sufficiently small, and both groups have at least 100 independently regenerated streams.

Otherwise it returns an explicit negative or inconclusive status. The exact economic constraints and bootstrap random seed are part of the protocol, not chosen after inspecting results.

In [ ]:
verdict = preregistered_verdict(models, economics, protocol)
display(verdict)
context = descriptive_context(models, economics)
display(context.round(4))
if SMOKE:
    assert set(verdict["status"]) == {"exploratory_insufficient_streams"}

## What the experiment can and cannot establish

The analysis can quantify outcomes under the predeclared synthetic data-generating process and detector specifications. It cannot demonstrate causal benefit for an actual agricultural or business intervention, cannot guarantee detector control under an unknown null process, and does not prove superiority over all conceivable tuning choices.

An observed false-alarm bound conditional on one calibrated threshold does not incorporate uncertainty across potential calibration datasets. Moreover, economic bootstrap intervals can be imprecise for skewed or heavy-tailed paired regret differences. Larger independent replication, calibration-seed sensitivity and preregistered external tasks are natural next steps.